In [1]:
from brocode.skills.utils import _load_skill, load_skill_body, load_script, load_reference, load_asset, list_skills, load_skill_body
from pathlib import Path
from functools import partial

In [2]:
ROOT = Path.cwd().resolve().parent
SKILL_DIR = ROOT / "skills"

In [3]:
load_skill = partial(_load_skill, path=SKILL_DIR)

In [4]:
skill = load_skill("read-file")

In [5]:
skill

Skill(name='read-file', description='use this skill only when the person wanna read any type of file', version='v0.1.0', path=WindowsPath('D:/brocode/skills/read-file'), tags=['read', 'file'], keywords=['read file', 'read'], default=True, status='experimental')

In [6]:
load_skill_body(skill)

"# Read file\n\nwhen invoked, read and return the file's content"

In [7]:
load_script(skill)

[WindowsPath('D:/brocode/skills/read-file/scripts/read_directory.py'),
 WindowsPath('D:/brocode/skills/read-file/scripts/read_file.py')]

In [8]:
load_reference(skill)

[]

In [9]:
load_asset(skill)

[]

In [10]:
from pathlib import Path

ROOT = Path.cwd().resolve().parent
skills = list((ROOT / "skills").rglob("SKILL.md"))
skills

[WindowsPath('D:/brocode/skills/read-file/SKILL.md'),
 WindowsPath('D:/brocode/skills/tell-jokes/SKILL.md')]

In [11]:
[s.parent.name for s in skills]

['read-file', 'tell-jokes']

In [12]:
list_skills(path=ROOT)

[Skill(name='read-file', description='use this skill only when the person wanna read any type of file', version='v0.1.0', path=WindowsPath('D:/brocode/skills/read-file'), tags=['read', 'file'], keywords=['read file', 'read'], default=True, status='experimental'),
 Skill(name='tell-jokes', description='Tell a joke when the user asks for one, or when they seem sad, frustrated, or stressed and could use a laugh', version='v0.1.0', path=WindowsPath('D:/brocode/skills/tell-jokes'), tags=['joke', 'funny', 'laugh'], keywords=['fun', 'funny', 'laugh', 'cheer up', 'lighten the mood'], default=False, status='stable')]

In [18]:
import argparse

def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--skill', type=str, required=True, help='skill path')
    return parser

parser = get_args()
parser._actions[1:]

[_StoreAction(option_strings=['--skill'], dest='skill', nargs=None, const=None, default=None, type=<class 'str'>, choices=None, required=True, help='skill path', metavar=None)]

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class Args:
    name:str
    type:str
    description:str
    required:bool

@dataclass
class Tool:
    name:str
    description:str
    args: List[Args]

In [22]:
import importlib.util
from pathlib import Path

ROOT = Path.cwd().resolve().parent
script_path = ROOT / "skills" / "read-file" / "scripts" / "read_file.py"

# 1. Build a module spec from the file path
spec = importlib.util.spec_from_file_location(script_path.stem, script_path)

# 2. Create an empty module object from that spec
module = importlib.util.module_from_spec(spec)

# 3. Actually run the file's top-level code, populating the module
spec.loader.exec_module(module)

parser = module.get_args()
parser._actions[1:]

[_StoreAction(option_strings=['--path'], dest='path', nargs=None, const=None, default=None, type=<class 'str'>, choices=None, required=True, help='a file path', metavar=None)]

In [25]:
ROOT = Path.cwd().resolve().parent
script_path = ROOT / "skills" / "read-file" / "scripts" / "read_file.py"

In [26]:
script_path.stem

'read_file'

In [1]:
from brocode.utils.tool_management import load_tool_schema
from pathlib import Path

In [ ]:
SCRIPT_DIR = Path.cwd().resolve().parent / "skills" / "read-file" / "scripts"
SCRIPT_DIR.is_dir()

True

In [3]:
tools = [load_tool_schema(SCRIPT_DIR / filename) for filename in ["read_file.py", "read_directory.py"]]

In [4]:
def register_tool(tools:list)->list:
    registered_tools = []
    for tool in tools:
        _tool = {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        arg.name: {
                            "type": arg.type,
                            "description": arg.description
                        } for arg in tool.args
                    },
                    "required": [arg.name for arg in tool.args if arg.required]
                }
            }
        }
        registered_tools.append(_tool)
    return registered_tools

In [5]:
register_tool(tools)

[{'type': 'function',
  'function': {'name': 'read_file',
   'description': 'Use this function when the person asks for reading only a single file',
   'parameters': {'type': 'object',
    'properties': {'path': {'type': 'string', 'description': 'a file path'}},
    'required': ['path']}}},
 {'type': 'function',
  'function': {'name': 'read_directory',
   'description': 'Use this function when the person asks for reading all files in a directory',
   'parameters': {'type': 'object',
    'properties': {'path': {'type': 'string',
      'description': 'a directory path'}},
    'required': ['path']}}}]